In [1]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,stats_test,STT_INIT_POINTS, STT_N_ITER, VOT_INIT_POINTS, VOT_N_ITER
from scipy import stats

VOT config

In [10]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : (0,7),
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': (-3, -0.001),
           'betaCostLow':(-0.2,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.2),
           'weightWalk': (0.5,3),
           'weightWait': (1,3),
           'weightFeeder': (0.5,3), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [11]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : (0,7),
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

In [38]:
stt_gen_synth_pop_baseline = [0.9513533794692444,
 0.9540499640034996,
 0.9532166953980288,
 0.9513134915454037,
 0.9512552091672533,
 0.9569671989366763,
 0.9528637863647696,
 0.9504651377198182,
 0.955995576349999,
 0.9522752271417112,
 0.9551395816499839,
 0.9553854591161569,
 0.9523419406392566,
 0.9552537633758673,
 0.9517056127575784,
 0.9519883920606923,
 0.9539855575549248,
 0.9522222328483616,
 0.9553597806303266,
 0.9536908172784654,
 0.9513282607052767,
 0.954356151933595,
 0.9528620778037029,
 0.9563416282312993,
 0.9496571492099861,
 0.9537171992317458,
 0.9541091332242164,
 0.9531096367142212,
 0.9565184545949811,
 0.9515004144918009]

vot_gen_synth_pop_baseline = [1.0929993810953729,
 1.092537993560709,
 1.0865640317206078,
 1.0959348706176169,
 1.0947350560748064,
 1.0927767325299085,
 1.0893223005965802,
 1.1003685747503171,
 1.0946657407223848,
 1.099645273873656,
 1.089415588688725,
 1.0971617454200902,
 1.0853587725518516,
 1.0912919174173208,
 1.0957690661414057,
 1.0944892365679983,
 1.094856614685757,
 1.0932321591556087,
 1.0985506426944947,
 1.0930877805767232,
 1.093250597527754,
 1.0969914034853432,
 1.0856382326567044,
 1.0961884816806202,
 1.0856580695199833,
 1.096235876394585,
 1.096103288647365,
 1.0951075295803878,
 1.0972218243244953,
 1.092757121017253]

stt_base_pop_baseline = [7.707496601071758,
 7.707436184480112,
 7.7073593723417755,
 7.708039607423291,
 7.705864289765003,
 7.709259901004068,
 7.707693312338857,
 7.70593735268445,
 7.706104494273714,
 7.704753692947246,
 7.7058294949652275,
 7.706207809803259,
 7.708250154429711,
 7.706808264388042,
 7.7076438703080825,
 7.7053095095734045,
 7.705551949630614,
 7.70548003397435,
 7.706312792905369,
 7.708235133983464,
 7.707889724324994,
 7.706785790604341,
 7.705627272353155,
 7.7076598846156745,
 7.706996388961764,
 7.708652864074839,
 7.708328253582076,
 7.708545033254906,
 7.705698322026233,
 7.705627426517699]

vot_base_pop_baseline = [7.752448578338818,
 7.753602605489025,
 7.754083366420699,
 7.757344985370771,
 7.755130558951879,
 7.753894394329194,
 7.753339969205057,
 7.754861805436822,
 7.758271964423312,
 7.757820741820273,
 7.755458507671064,
 7.7554577041457735,
 7.754894996183904,
 7.756599284766706,
 7.755770854609119,
 7.755384765321047,
 7.754860901248403,
 7.754544456678933,
 7.75172863728447,
 7.756875537766206,
 7.75649487712143,
 7.754669662701284,
 7.755538073875542,
 7.75407234682431,
 7.757303390731712,
 7.754888315023639,
 7.75757684580427,
 7.75689954879258,
 7.7530494519756825,
 7.755446078087407]

In [5]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



RQ3.1

In [6]:

rq_3_1_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-1.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-1.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-1.csv',
    "other_args" : "--use_random_seed true"
}
rq3_1_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_3_1_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq3_1_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.87860 | 0.0       | 2.2032449 | 0.0008006 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -14.18428 | 0.0       | -3.018985 | 5.6052119 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -23.87463 | 0.0   

In [7]:
rq_3_1_config_no_seed = rq_3_1_config
rq_3_1_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_1_results = []
for i in range(0,30):
    rq_3_1_results.append(-call_stt_simulation(config=rq_3_1_config_no_seed,**rq3_1_optimiser.max['params']))


In [8]:
rq_3_1_results

[7.637922456693038,
 7.637493209244336,
 7.636253825214027,
 7.635788469806259,
 7.636807891525833,
 7.636261499187812,
 7.63656752367518,
 7.633909069718626,
 7.635839894968236,
 7.635376563146373,
 7.636271792874385,
 7.635503460369104,
 7.63473489255757,
 7.637779683768454,
 7.63634332915439,
 7.636344983359584,
 7.636129963580441,
 7.635212050137168,
 7.638382849807827,
 7.63674128136572,
 7.636930084574709,
 7.637243175004101,
 7.635754848219434,
 7.635557406172063,
 7.636101926723893,
 7.634907023480373,
 7.637241326518251,
 7.63648774845832,
 7.638005013881438,
 7.635218302073165]

In [26]:
np.mean(rq_3_1_results)

np.float64(7.636303718175335)

In [27]:
np.std(rq_3_1_results)

np.float64(0.0010258093187313333)

In [37]:
rq3_1_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-1.0206911301913568),
 'alphaCarDriver': np.float64(5.474104974931597),
 'alphaCarPassenger': np.float64(4.117908802762887),
 'alphaBus': np.float64(-0.1753816317949405),
 'alphaTrain': np.float64(-0.12582147791920983),
 'betaTimeWalk': np.float64(-0.04),
 'betaTimeBike': np.float64(-0.03),
 'betaTimeCarDriver': np.float64(-0.02),
 'betaTimeCarPassenger': np.float64(-0.02),
 'betaTimeBus': np.float64(-0.02),
 'betaTimeTrain': np.float64(-0.02),
 'betaCostCarDriver': np.float64(-0.15),
 'betaCostCarPassenger': np.float64(-0.15),
 'betaCostBus': np.float64(-0.1),
 'betaCostTrain': np.float64(-0.1),
 'betaTimeWalkTransport': np.float64(-0.03),
 'betaChangesTransport': np.float64(-0.3)}

RQ3.2

In [12]:

rq_3_2_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-2.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-2.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-2.csv',
    "other_args" : "--use_random_seed true"
}
rq_3_2_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_3_2_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq_3_2_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.65582 | 0.0       | 2.2032449 | 0.0008006 | -1.976674 | -3.532441 | -4.076614 | -2.441405 | -0.2      | -0.2      | -0.2      | 1.5479862 | 2.3704390 | 1.0111306 | 1.0       | 1.0       |
| 2         | -13.42433 | 0.0       | -0.826951 | 3.9108287 | -3.596130 | -3.018985 | 3.0074456 | -0.096183 | -0.2      | -0.2      | -0.2      | 2.7365166 | 1.1700884 | 0.5976369 | 1.0       | 1.0       |
| 3         | -10.50232 | 0.0       | -0.788923 | 6.7052267 | 0.3316528 | 1.9187711 | -1.844843 | -0.941183 | -0.2      | -0.2      | -0.2      | 2.9721527 | 2.4963313 | 1.2011

In [13]:
rq_3_2_config_no_seed = rq_3_2_config
rq_3_2_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_2_results = []
for i in range(0,30):
    rq_3_2_results.append(- call_vot_simulation(config=rq_3_2_config_no_seed,**rq_3_2_optimiser.max['params']))

In [14]:
rq_3_2_results

[7.732246451147498,
 7.728016046394473,
 7.73420483249664,
 7.730697670403327,
 7.729351702271057,
 7.731953814471121,
 7.73096174183414,
 7.73184996895674,
 7.731391705778112,
 7.7305152640982175,
 7.733483143664493,
 7.728351975998814,
 7.728288300023749,
 7.732711872454163,
 7.726978879470738,
 7.7337790962694255,
 7.730580393845243,
 7.7293540982667555,
 7.730649845261015,
 7.731817375974287,
 7.733483536566635,
 7.734224139144694,
 7.721608319552379,
 7.726724047508985,
 7.727794510763305,
 7.736145259968143,
 7.728170135356154,
 7.731732381155161,
 7.7347211814582595,
 7.729094853318016]

In [28]:
np.mean(rq_3_2_results)

np.float64(7.730696084795725)

In [29]:
np.std(rq_3_2_results)

np.float64(0.0029418759894615)

In [36]:
rq_3_2_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-0.45397711769983595),
 'alphaCarDriver': np.float64(7.0),
 'alphaCarPassenger': np.float64(5.0),
 'alphaBus': np.float64(3.9824780886950855),
 'alphaTrain': np.float64(5.0),
 'betaChangesTransport': np.float64(-0.001),
 'betaCostLow': np.float64(-0.2),
 'betaCostMed': np.float64(-0.2),
 'betaCostHigh': np.float64(-0.2),
 'weightWalk': np.float64(3.0),
 'weightWait': np.float64(1.7265129599855593),
 'weightFeeder': np.float64(3.0),
 'weightVotCosts': np.float64(1.0),
 'weightTangibleCosts': np.float64(1.0)}

RQ3.3

In [15]:

rq_3_3_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-3.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-3.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-3.csv',
    "other_args" : "--use_random_seed true"
}
rq3_3_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_3_3_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq3_3_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.54190 | 0.0       | 2.2032449 | 0.0008006 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -9.003683 | 0.0       | -3.018985 | 5.6052119 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -22.97631 | 0.0   

In [16]:
rq_3_config_no_seed = rq_3_3_config
rq_3_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_3_results = []
for i in range(0,30):
    rq_3_3_results.append(-call_stt_simulation(config=rq_3_config_no_seed,**rq3_3_optimiser.max['params']))

In [17]:
rq_3_3_results

[0.9872265263932241,
 0.975387505632247,
 0.979774474023441,
 0.9611706315615299,
 0.9758225496821942,
 0.9912720222191524,
 0.9786472622977728,
 0.9707734556408016,
 0.9589614522953824,
 0.9778091207586675,
 0.9938041005830743,
 0.9785375954695082,
 0.9800402754939741,
 0.9803481609916184,
 0.9630769294719718,
 0.970431521852212,
 0.9871907479925794,
 0.9847722415992504,
 0.9667470793698894,
 0.9791259656232342,
 0.967223795174644,
 0.9830574938596977,
 0.9789549393146224,
 0.9778277545209534,
 0.9791771857495931,
 0.9856455297208495,
 0.9994374199915949,
 0.9956614530663627,
 0.9707636717938959,
 0.9716262701354403]

In [30]:
np.mean(rq_3_3_results)

np.float64(0.9783431710759791)

In [31]:
np.std(rq_3_3_results)

np.float64(0.009722378147752403)

In [35]:
rq3_3_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-1.3045068043690455),
 'alphaCarDriver': np.float64(6.288321379306041),
 'alphaCarPassenger': np.float64(5.0),
 'alphaBus': np.float64(-0.8872205029845929),
 'alphaTrain': np.float64(-2.8543521300591355),
 'betaTimeWalk': np.float64(-0.04),
 'betaTimeBike': np.float64(-0.03),
 'betaTimeCarDriver': np.float64(-0.02),
 'betaTimeCarPassenger': np.float64(-0.02),
 'betaTimeBus': np.float64(-0.02),
 'betaTimeTrain': np.float64(-0.02),
 'betaCostCarDriver': np.float64(-0.15),
 'betaCostCarPassenger': np.float64(-0.15),
 'betaCostBus': np.float64(-0.1),
 'betaCostTrain': np.float64(-0.1),
 'betaTimeWalkTransport': np.float64(-0.03),
 'betaChangesTransport': np.float64(-0.3)}

RQ3.4

In [18]:
rq_3_4_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-4.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-4.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-4.csv',
    "other_args" : "--use_random_seed true"
}
rq_3_4_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_3_4_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq_3_4_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.32425 | 0.0       | 2.2032449 | 0.0008006 | -1.976674 | -3.532441 | -4.076614 | -2.441405 | -0.2      | -0.2      | -0.2      | 1.5479862 | 2.3704390 | 1.0111306 | 1.0       | 1.0       |
| 2         | -10.30755 | 0.0       | -0.826951 | 3.9108287 | -3.596130 | -3.018985 | 3.0074456 | -0.096183 | -0.2      | -0.2      | -0.2      | 2.7365166 | 1.1700884 | 0.5976369 | 1.0       | 1.0       |
| 3         | -5.627628 | 0.0       | -0.788923 | 6.7052267 | 0.3316528 | 1.9187711 | -1.844843 | -0.941183 | -0.2      | -0.2      | -0.2      | 2.9721527 | 2.4963313 | 1.2011

In [19]:
rq_3_4_config_no_seed = rq_3_4_config
rq_3_4_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_4_results = []
for i in range(0,30):
    rq_3_4_results.append(- call_vot_simulation(config=rq_3_4_config_no_seed,**rq_3_4_optimiser.max['params']))

In [20]:
rq_3_4_results

[1.101821252677569,
 1.0982015460260661,
 1.1058078477192426,
 1.0952592250736255,
 1.105624394949262,
 1.1018182627571131,
 1.0948763928463632,
 1.1118192668585622,
 1.1050109865643623,
 1.0983044664764925,
 1.0864736238789952,
 1.1034659827911992,
 1.0929597738056973,
 1.098772260850749,
 1.0990555174129342,
 1.095706134103142,
 1.1020791196124422,
 1.094949659013181,
 1.0976175784089741,
 1.1019065082430037,
 1.1057048985877123,
 1.1034282942467968,
 1.0977110368891176,
 1.083442364206457,
 1.089925247194916,
 1.1009215981807592,
 1.0967263100054632,
 1.1013027083024258,
 1.0958369245095771,
 1.095871323279065]

In [32]:
np.mean(rq_3_4_results)

np.float64(1.098746683515709)

In [33]:
np.std(rq_3_4_results)

np.float64(0.005836255145550286)

In [34]:
rq_3_4_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-0.5248702783882966),
 'alphaCarDriver': np.float64(6.467213899471103),
 'alphaCarPassenger': np.float64(5.0),
 'alphaBus': np.float64(3.7643831228533333),
 'alphaTrain': np.float64(2.3929046975279853),
 'betaChangesTransport': np.float64(-2.1123638261983073),
 'betaCostLow': np.float64(-0.2),
 'betaCostMed': np.float64(-0.2),
 'betaCostHigh': np.float64(-0.2),
 'weightWalk': np.float64(3.0),
 'weightWait': np.float64(2.2724305099852056),
 'weightFeeder': np.float64(3.0),
 'weightVotCosts': np.float64(1.0),
 'weightTangibleCosts': np.float64(1.0)}

Significance

In [39]:
stats_test(stt_base_pop_baseline, rq_3_1_results)

P-value: 6.73335701086599e-89
Statistically significant difference.
t-statistic: 242.5390
p-value: 0.0000
95% CI of the difference: [0.0700, 0.0712]


In [40]:
stats_test(vot_base_pop_baseline, rq_3_2_results)

P-value: 1.180049410592238e-43
Statistically significant difference.
t-statistic: 39.5507
p-value: 0.0000
95% CI of the difference: [0.0233, 0.0258]


In [41]:
stats_test(stt_gen_synth_pop_baseline, rq_3_3_results)

P-value: 1.111276990111967e-19
Statistically significant difference.
t-statistic: -13.5918
p-value: 0.0000
95% CI of the difference: [-0.0287, -0.0213]


In [42]:
stats_test(vot_gen_synth_pop_baseline, rq_3_4_results)

P-value: 0.00022675731152453616
Statistically significant difference.
t-statistic: -3.9328
p-value: 0.0002
95% CI of the difference: [-0.0078, -0.0025]


In [25]:
rq3_3_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-1.3045068043690455),
 'alphaCarDriver': np.float64(6.288321379306041),
 'alphaCarPassenger': np.float64(5.0),
 'alphaBus': np.float64(-0.8872205029845929),
 'alphaTrain': np.float64(-2.8543521300591355),
 'betaTimeWalk': np.float64(-0.04),
 'betaTimeBike': np.float64(-0.03),
 'betaTimeCarDriver': np.float64(-0.02),
 'betaTimeCarPassenger': np.float64(-0.02),
 'betaTimeBus': np.float64(-0.02),
 'betaTimeTrain': np.float64(-0.02),
 'betaCostCarDriver': np.float64(-0.15),
 'betaCostCarPassenger': np.float64(-0.15),
 'betaCostBus': np.float64(-0.1),
 'betaCostTrain': np.float64(-0.1),
 'betaTimeWalkTransport': np.float64(-0.03),
 'betaChangesTransport': np.float64(-0.3)}